In [1]:
import requests
import pandas as pd
import uuid
import json
session = requests.Session()
medidas = [
    "%5BMeasures%5D.%5BVENTAS%20LOCALES%20GRAVADAS%20(411)%5D",
    "%5BMeasures%5D.%5BVENTAS%20ACTIVOS%20FIJOS%20GRAVADOS%20(412)%5D",
    "%5BMeasures%5D.%5BVENTAS%20LOCALES%200%25%20(413)%5D",
    "%5BMeasures%5D.%5BVENTAS%200%25%20DE%20ACTIVOS%20FIJOS%20(414)%5D",
    "%5BMeasures%5D.%5BVENTAS%200%25%20CON%20CRED%20TRIB%20(415)%5D",
    "%5BMeasures%5D.%5BVENTAS%200%25%20DE%20AF%20CON%20CRED%20TRIB%20(416)%5D",
    "%5BMeasures%5D.%5BVENTAS%20LOCALES%20TARIFA%20VARIABLE%20(420)%5D",
    "%5BMeasures%5D.%5BVENTAS%20LOCALES%205%25%20(435)%5D"
]

df_filas = pd.DataFrame({
    "tabla": [           
        "TIPO%20CONTRIBUYENTE",
        "GRAN%20CONTRIBUYENTE",
        "CLASE%20CONTRIBUYENTE",
        "ACTIVIDAD%20ECONOMICA",
        "ACTIVIDAD%20ECONOMICA",
        "UBICACION%20GEOGRAFICA"
    ],
    "fila": [
     
        "%5BTIPO%20CONTRIBUYENTE%5D/%5BTIPO%20CONTRIBUYENTE%5D.%5BTIPO%20CONTRIBUYENTE%5D",
        "%5BGRAN%20CONTRIBUYENTE%5D/%5BGRAN%20CONTRIBUYENTE%5D.%5BGRAN%20CONTRIBUYENTE%5D",
        "%5BCLASE%20CONTRIBUYENTE%5D/%5BCLASE%20CONTRIBUYENTE%5D.%5BCLASE%20CONTRIBUYENTE%5D",
        "%5BACTIVIDAD%20ECONOMICA%5D/%5BACTIVIDAD%20ECONOMICA%5D.%5BCODIGO_OPERA_ACTIVIDAD_ECO%5D",
        "%5BACTIVIDAD%20ECONOMICA%5D/%5BACTIVIDAD%20ECONOMICA%5D.%5BCODIGO_OPERA_FAMILIA%5D",
        "%5BUBICACION%20GEOGRAFICA%5D/%5BUBICACION%20GEOGRAFICA%5D.%5BCANTON%5D"
    ]
})

In [2]:
VAL_REQ = str(uuid.uuid4()).upper()

### AQUI COMENZAMOS A SELECCIONAR EL CUBO QUE USAREMOS USO DE EJEMPLO

Xurl = f"https://srienlinea.sri.gob.ec/saiku/rest/saiku/anonymousUser/query/{VAL_REQ}"

headers = {
    "Accept": "application/json",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "Content-Type": "application/x-www-form-urlencoded",
    "Referer": "https://srienlinea.sri.gob.ec/saiku-ui/",
    "User-Agent": "Mozilla/5.0"
}

data = {
    "connection": "declaracion104",
    "catalog": "Declaracion",
    "schema": "Declaracion",
    "cube": "D104",
    "type": "QM",
    "formatter": "flattened"
}

#response = requests.post(url, headers=headers, data=data)

session.post(Xurl, headers=headers, data=data)

<Response [200]>

In [3]:
#### agregar  filtros anio mes

tabla = "ANIO%20FISCAL"
filtro = "%5BANIO%20FISCAL%5D/%5BANIO%20FISCAL%5D.%5BMES%20FISCAL%5D"
sub_filtro = "%5BANIO%20FISCAL%5D.%5BMES%20FISCAL%5D"

url = f"{Xurl}/axis/FILTER/dimension/{tabla}/hierarchy/{filtro}"

data = {
    "position": "0"
}

session.post(url, headers=headers, data=data)

####### SELECCION DE ULTIMO PERIODO DISPONIBLE

url = f"{Xurl}/result/metadata/dimensions/{tabla}/hierarchies/%5BANIO%20FISCAL%5D/levels/{sub_filtro}"

response = session.get(url, headers=headers)

data = response.json()


anios_meses = []
for item in data:
    unique_name = item.get("uniqueName", "")
    parts = unique_name.strip("[]").split("].[")
    if len(parts) >= 3:
        try:
            anio = int(parts[1])
            mes = int(parts[2])
            anios_meses.append((anio, mes))
        except ValueError:
            continue

ultimo_anio, ultimo_mes = max(anios_meses)            
            
#######


url2 = f"{Xurl}/axis/FILTER/dimension/{tabla}"

selections = [
    {
        "hierarchy": "[ANIO FISCAL]",
        "uniquename": "[ANIO FISCAL].[MES FISCAL]",
        "type": "level",
        "action": "delete"
    },
    {
        "uniquename": f"[ANIO FISCAL].[{ultimo_anio}].[{ultimo_mes}]",
        "type": "member",
        "action": "add"
    }
]

data = {
    "selections": json.dumps(selections)
}


session.put(url2, headers=headers, data=data)

<Response [200]>

In [4]:
### AGREGAR MEDIDAS

for i, medida in enumerate(medidas):

    url = f"{Xurl}/axis/COLUMNS/dimension/Measures/member/{medida}"

    data = {
        #"position": str(i)
        "position": "0"
        
    }

    response = session.post(url, headers=headers, data=data)

    print("medida:", medida)
    print("status:", response.status_code)

medida: %5BMeasures%5D.%5BVENTAS%20LOCALES%20GRAVADAS%20(411)%5D
status: 201
medida: %5BMeasures%5D.%5BVENTAS%20ACTIVOS%20FIJOS%20GRAVADOS%20(412)%5D
status: 201
medida: %5BMeasures%5D.%5BVENTAS%20LOCALES%200%25%20(413)%5D
status: 201
medida: %5BMeasures%5D.%5BVENTAS%200%25%20DE%20ACTIVOS%20FIJOS%20(414)%5D
status: 201
medida: %5BMeasures%5D.%5BVENTAS%200%25%20CON%20CRED%20TRIB%20(415)%5D
status: 201
medida: %5BMeasures%5D.%5BVENTAS%200%25%20DE%20AF%20CON%20CRED%20TRIB%20(416)%5D
status: 201
medida: %5BMeasures%5D.%5BVENTAS%20LOCALES%20TARIFA%20VARIABLE%20(420)%5D
status: 201
medida: %5BMeasures%5D.%5BVENTAS%20LOCALES%205%25%20(435)%5D
status: 201


In [5]:
#### AGREGAR FILAS

for _, row in df_filas.iterrows():

    tabla = row["tabla"]
    columna = row["fila"]
    #position = row["position"]

    url = f"{Xurl}/axis/ROWS/dimension/{tabla}/hierarchy/{columna}"

    data = {
        #"position": str(position)
        "position": "0"
    }

    response = session.post(url, headers=headers, data=data)

    print(tabla,  response.status_code)

TIPO%20CONTRIBUYENTE 201
GRAN%20CONTRIBUYENTE 201
CLASE%20CONTRIBUYENTE 201
ACTIVIDAD%20ECONOMICA 201
ACTIVIDAD%20ECONOMICA 201
UBICACION%20GEOGRAFICA 201


In [6]:
#### OBETENER LOS RESULTADOS DEL QUERY ARMADO

url = f"{Xurl}/result/flattened"

response = session.get(url, headers=headers)

data = response.json()["cellset"]

columns = [c["value"] for c in data[0]]

rows = []

for r in data[1:]:

    fila = []

    for c in r:

        if c["type"] == "DATA_CELL":
            fila.append(float(c["properties"]["raw"]))
        else:
            fila.append(c["value"])

    rows.append(fila)

df = pd.DataFrame(rows, columns=columns)

df = df.replace("null", None)
df = df.ffill()

#EXPORTAR A EXCEL
df.to_excel(f"DATA_SRI_{ultimo_anio}-{ultimo_mes}.xlsx", index=False)